# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses *(À compléter)* :**
- **Test #1 :** $H_0 =$ `...` / $H_1 =$ `...` *(Justification du choix du test : `...`)*
- **Test #2 :** $H_0 =$ `...` / $H_1 =$ `...` *(Justification du choix du test : `...`)*

In [ ]:
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"
df_silver = spark.read.format("delta").load(SILVER_MAIN_PATH)

# Échantillon des commandes livrées avec les variables nécessaires aux tests.
df_sample = (
    df_silver
    .filter(
        F.col("delivery_days").isNotNull()
        & F.col("delay_days").isNotNull()
        & F.col("order_value").isNotNull()
    )
    .select("delivery_days", "delay_days", "order_value", "freight_value",
            "item_count", "review_score", "customer_state", "is_late")
    .dropna(subset=["delivery_days", "delay_days", "order_value"])
    .limit(30000)
    .toPandas()
)

# Test 1 : comparaison des délais de livraison entre commandes à forte et faible valeur.
median_value = df_sample["order_value"].median()
group_low = df_sample.loc[df_sample["order_value"] <= median_value, "delivery_days"]
group_high = df_sample.loc[df_sample["order_value"] > median_value, "delivery_days"]
stat_1, p_val_1 = stats.mannwhitneyu(group_low, group_high, alternative="two-sided")
print(f"Mann-Whitney U : statistique={stat_1:.3f}, p-value={p_val_1:.6g}")
print("Interprétation :", "différence statistiquement significative" if p_val_1 < 0.05 else "pas de différence significative au seuil de 5 %")

# Test 2 : corrélation monotone entre frais de port et délai de livraison.
rho, p_val_2 = stats.spearmanr(df_sample["freight_value"], df_sample["delivery_days"])
print(f"Spearman : rho={rho:.3f}, p-value={p_val_2:.6g}")
print("Interprétation :", "corrélation significative" if p_val_2 < 0.05 else "corrélation non significative au seuil de 5 %")

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [ ]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/customer_segments"

# Agrégation par client : volume, valeur et délai moyen des commandes.
entities_pd = (
    df_silver
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.avg("order_value").alias("average_order_value"),
        F.avg("delivery_days").alias("average_delivery_days"),
        F.avg("delay_days").alias("average_delay_days"),
        F.avg("is_late").alias("late_rate")
    )
    .filter(F.col("customer_unique_id").isNotNull())
    .dropna()
    .limit(50000)
    .toPandas()
)

feature_cols = ["orders_count", "average_order_value", "average_delivery_days", "late_rate"]
X = entities_pd[feature_cols].clip(lower=0)
X_log = np.log1p(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Nommer les segments selon leur fréquence de commande et leur taux de retard.
cluster_profile = entities_pd.groupby("cluster_id")[["orders_count", "late_rate"]].median()
order_rank = cluster_profile["orders_count"].rank(method="first")
late_rank = cluster_profile["late_rate"].rank(method="first")
labels = {}
for cluster_id in cluster_profile.index:
    if order_rank[cluster_id] == order_rank.max():
        labels[cluster_id] = "Clients fidèles"
    elif late_rank[cluster_id] == late_rank.max():
        labels[cluster_id] = "Clients exposés aux retards"
    else:
        labels[cluster_id] = "Clients occasionnels"

entities_pd["segment"] = entities_pd["cluster_id"].map(labels)

(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Segmentation enregistrée : {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} clients)")
print(entities_pd.groupby("segment")[feature_cols].median())

## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

## 3. Modèle supervisé : prédiction d'une livraison en retard

Le modèle prédit `is_late` à partir d'informations disponibles au moment de la commande. La durée réelle de livraison et le retard réel ne sont pas utilisés comme variables explicatives afin d'éviter la fuite de données.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Variables disponibles au moment de l'achat uniquement.
model_df = (
    df_silver
    .filter(F.col("is_late").isNotNull())
    .select(
        F.col("is_late").cast("double").alias("label"),
        F.coalesce(F.col("order_value"), F.lit(0.0)).alias("order_value"),
        F.coalesce(F.col("freight_value"), F.lit(0.0)).alias("freight_value"),
        F.coalesce(F.col("item_count"), F.lit(1)).cast("double").alias("item_count"),
        F.month("order_purchase_timestamp").cast("double").alias("purchase_month"),
        F.dayofweek("order_purchase_timestamp").cast("double").alias("purchase_weekday"),
        F.coalesce(F.col("customer_state"), F.lit("UNKNOWN")).alias("customer_state")
    )
    .dropna()
)

state_indexer = StringIndexer(
    inputCol="customer_state", outputCol="customer_state_index", handleInvalid="keep"
)
state_encoder = OneHotEncoder(
    inputCols=["customer_state_index"], outputCols=["customer_state_vec"]
)
assembler = VectorAssembler(
    inputCols=[
        "order_value", "freight_value", "item_count",
        "purchase_month", "purchase_weekday", "customer_state_vec"
    ],
    outputCol="features"
)
classifier = RandomForestClassifier(
    labelCol="label", featuresCol="features",
    numTrees=40, maxDepth=7, seed=42
)
pipeline = Pipeline(stages=[state_indexer, state_encoder, assembler, classifier])

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
).evaluate(predictions)
print(f"AUC ROC : {auc:.3f}")
print(f"F1-score : {f1:.3f}")
predictions.select("label", "prediction", "probability").show(10, truncate=False)

# Sauvegarde du modèle Spark pour assurer sa traçabilité.
MODEL_PATH = "/lakehouse/gold/late_delivery_random_forest"
model.write().overwrite().save(MODEL_PATH)
print("Modèle sauvegardé :", MODEL_PATH)

In [ ]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")